# Parte III — Evaluación e interpretación de recomendaciones

Esta comparación usa una validación de 2023 para seleccionar ALS y una prueba retrospectiva de 2024. Los candidatos no observados no son negativos confirmados, las relaciones no son preferencias y los releases actuales no prueban disponibilidad histórica. Las métricas solo incluyen entidades con positivos elegibles; no describen proveedores o entidades fríos.

In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import pyarrow.dataset as ds
from IPython.display import display
ROOT = Path.cwd()
if not (ROOT / 'docs').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.features.build_part1 import directory_fingerprint
source = ROOT / 'data/processed/recommendation'
events = ROOT / 'data/processed/fact_award_supplier'
folders = {name: ROOT / 'artifacts' / folder for name, folder in {
    'popularidad': 'recommendation_baseline', 'vecindario': 'recommendation_neighbors',
    'als': 'recommendation_als', 'prueba': 'recommendation_test'}.items()}
assert (source / '_SUCCESS').is_file()
fingerprint = directory_fingerprint(source)
source_meta = json.loads((source / '_metadata.json').read_text(encoding='utf-8'))
assert directory_fingerprint(events) == source_meta['input_sha256'], 'Cambió la fuente de eventos'
metadata = {}
for name, path in folders.items():
    assert (path / '_SUCCESS').is_file(), f'Falta publicación: {path}'
    metadata[name] = json.loads((path / '_metadata.json').read_text(encoding='utf-8'))
    assert metadata[name]['input_sha256'] == fingerprint, f'Cambiaron los cortes: {name}'
print('Huella de cortes:', fingerprint)
print('ALS seleccionado en validación:', metadata['als']['selected'])

Huella de cortes: 64ec49417ae4a8b0729709e358b4860be3c88574da6f9d273b0e65764cee135b
ALS seleccionado en validación: {'rank': 16, 'reg': 0.1, 'alpha': 10.0}


## Comparación de métricas y cobertura

La selección se congeló por NDCG@10 de validación. La prueba se usa para medir una vez esa elección. Las métricas se promedian por entidad y las posiciones ausentes cuentan como fallos.

In [2]:
rows=[]
for period in ('validación 2023', 'prueba 2024'):
    for method in ('popularidad', 'vecindario', 'als'):
        if period.startswith('validación'):
            meta = metadata[method]
            if method == 'als':
                chosen = next(r for r in meta['results'] if r['config'] == meta['selected'])
                scores = chosen['metrics_macro']; coverage = chosen['provider_coverage']
                nonempty = meta['entities_evaluated']
            else:
                scores = meta['metrics_macro']; coverage = meta['provider_coverage']
                nonempty = meta.get('entities_with_ranking', meta['entities_evaluated'])
        else:
            entry = metadata['prueba']['metrics_macro'][method]
            scores = entry; coverage = entry['provider_coverage']; nonempty = entry['entities_with_ranking']
        for k in ('10','20'):
            rows.append({'período':period, 'método':method, 'K':int(k), **scores[k],
                         'entidades con ranking @20':nonempty, 'cobertura proveedores @20':coverage})
display(pd.DataFrame(rows).set_index(['período','método','K']).round(4))

recall  precision    ndcg  \
período         método      K                               
validación 2023 popularidad 10  0.0257     0.0143  0.0217   
                            20  0.0342     0.0100  0.0238   
                vecindario  10  0.0232     0.0179  0.0282   
                            20  0.0335     0.0129  0.0297   
                als         10  0.0555     0.0295  0.0558   
                            20  0.0760     0.0213  0.0607   
prueba 2024     popularidad 10  0.0303     0.0133  0.0252   
                            20  0.0424     0.0090  0.0282   
                vecindario  10  0.0544     0.0276  0.0554   
                            20  0.0762     0.0201  0.0606   
                als         10  0.0748     0.0340  0.0657   
                            20  0.1087     0.0263  0.0754   

                                entidades con ranking @20  \
período         método      K                               
validación 2023 popularidad 10                       1843   
                            20                       1843   
                vecindario  10                       1607   
                            20                       1607   
                als         10                       1843   
                            20                       1843   
prueba 2024     popularidad 10                       1879   
                            20                       1879   
                vecindario  10                       1785   
                            20                       1785   
                als         10                       1879   
                            20                       1879   

                                cobertura proveedores @20  
período         método      K                              
validación 2023 popularidad 10                     0.0012  
                            20                     0.0012  
                vecindario  10                     0.3353  
                            20                     0.3353  
                als         10                     0.0436  
                            20                     0.0436  
prueba 2024     popularidad 10                     0.0006  
                            20                     0.0006  
                vecindario  10                     0.2187  
                            20                     0.2187  
                als         10                     0.0207  
                            20                     0.0207

## Distribución y segmentos de historial en prueba

Los tramos se calculan a partir del número de proveedores distintos vistos por la entidad en el entrenamiento de prueba. Muestran heterogeneidad y tamaño, no una comparación causal. La evaluación excluye actores fríos.

In [3]:
metrics = pd.read_parquet(folders['prueba'] / 'metrics_by_entity.parquet')
train = ds.dataset(source / '2024/train', format='parquet', exclude_invalid_files=True).to_table(columns=['entity_id','supplier_id']).to_pandas()
history = train.groupby('entity_id').supplier_id.nunique().rename('historial')
analysis = metrics.merge(history, on='entity_id', validate='many_to_one')
analysis['tramo'] = pd.cut(analysis.historial, bins=[0,5,20,float('inf')], labels=['1–5','6–20','21+'])
display(analysis.loc[analysis.k.eq(10)].groupby(['method','tramo'], observed=True).agg(
    entidades=('entity_id','size'), recall=('recall','mean'), ndcg=('ndcg','mean'),
    longitud=('ranking_length','mean')).round(4))
display(analysis.loc[analysis.k.eq(10)].groupby('method')[['recall','ndcg']].quantile([.25,.5,.75]).round(4))

entidades  recall    ndcg  longitud
method      tramo                                     
als         1–5          303  0.0734  0.0458   10.0000
            6–20         698  0.0795  0.0595   10.0000
            21+          878  0.0716  0.0775   10.0000
popularidad 1–5          303  0.0327  0.0180   10.0000
            6–20         698  0.0318  0.0250   10.0000
            21+          878  0.0282  0.0278   10.0000
vecindario  1–5          303  0.0407  0.0336    7.3333
            6–20         698  0.0591  0.0504    9.7564
            21+          878  0.0554  0.0669    9.9886

recall    ndcg
method                          
als         0.25  0.0000  0.0000
            0.50  0.0000  0.0000
            0.75  0.0308  0.0649
popularidad 0.25  0.0000  0.0000
            0.50  0.0000  0.0000
            0.75  0.0000  0.0000
vecindario  0.25  0.0000  0.0000
            0.50  0.0000  0.0000
            0.75  0.0000  0.0000

## Incertidumbre por entidad

Intervalos bootstrap del 95 % de NDCG@10 con 1 000 remuestreos y semilla 42. La comparación pareada usa las mismas entidades. Los intervalos no corrigen cobertura, sesgo temporal, identidad ni dependencia entre entidades.

In [4]:
wide = metrics.loc[metrics.k.eq(10)].pivot(index='entity_id', columns='method', values='ndcg').sort_index()
assert wide.notna().all().all()
rng = np.random.default_rng(42)
indices = rng.integers(0, len(wide), size=(1000, len(wide)))
boot=[]
for label, values in [('popularidad',wide.popularidad.to_numpy()), ('vecindario',wide.vecindario.to_numpy()),
                      ('als',wide.als.to_numpy()), ('als − popularidad', (wide.als-wide.popularidad).to_numpy()),
                      ('als − vecindario', (wide.als-wide.vecindario).to_numpy())]:
    means=values[indices].mean(axis=1)
    boot.append({'comparación':label,'media':values.mean(),'IC 2,5 %':np.quantile(means,.025),'IC 97,5 %':np.quantile(means,.975)})
display(pd.DataFrame(boot).set_index('comparación').round(4))

,media,"IC 2,5 %","IC 97,5 %"
comparación,,,
popularidad,0.0252,0.0210,0.0291
vecindario,0.0554,0.0492,0.0615
als,0.0657,0.0589,0.0723
als − popularidad,0.0405,0.0347,0.0462
als − vecindario,0.0103,0.0035,0.0172


## Concentración e interpretación

La popularidad de un proveedor es el número de entidades distintas que lo adjudicaron en entrenamiento. El resumen usa las listas nativas hasta 20; una lista vacía no se rellena. Los valores altos no demuestran calidad ni sustitución entre proveedores.

In [5]:
ranks = pd.read_parquet(folders['prueba'] / 'rankings.parquet')
pop = train.groupby('supplier_id').entity_id.nunique().rename('entidades_históricas')
joined = ranks.merge(pop, on='supplier_id', validate='many_to_one')
display(joined.groupby('method').agg(recomendaciones=('supplier_id','size'),
    proveedores_distintos=('supplier_id','nunique'), popularidad_mediana=('entidades_históricas','median'),
    popularidad_media=('entidades_históricas','mean')).round(2))
print('La evidencia representa coocurrencia histórica; no demuestra que los proveedores ofrezcan productos equivalentes.')

,recomendaciones,proveedores_distintos,popularidad_mediana,popularidad_media
method,,,,
als,37580,985,27.0,36.69
popularidad,37580,29,55.0,71.52
vecindario,35460,10402,4.0,13.94


La evidencia representa coocurrencia histórica; no demuestra que los proveedores ofrezcan productos equivalentes.


## Límites para el cierre

La fuente puede omitir relaciones, los identificadores pueden estar fragmentados, y la ventana retrospectiva incorpora registros cuya disponibilidad histórica no se demostró. No se evaluó arranque en frío, idoneidad de proveedores, ni una recomendación basada en contenido; la semejanza colaborativa no acredita sustitución por categoría. Las métricas no son garantía de rendimiento futuro.